# Compare Submission Predictions by Market

This notebook compares two submission files market-by-market:
- run token A: `20-151106`
- run token B: `20-202340`

It resolves run tokens to `runs/<run_id>/submission.csv`, merges with test market labels, and reports:
- overall prediction deltas
- per-market summary metrics
- per-market overlay plots
- largest disagreements per market


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [ ]:
RUNS_DIR = Path("runs")
TEST_PATH = Path("data/test_for_participants.csv")

RUN_A_TOKEN = "20-151106"
RUN_B_TOKEN = "20-202340"


In [ ]:
def resolve_submission_source(token: str, runs_dir: Path = RUNS_DIR) -> Path:
    candidate = Path(token)
    if candidate.is_file():
        return candidate

    direct_run_submission = runs_dir / token / "submission.csv"
    if direct_run_submission.is_file():
        return direct_run_submission

    matches = [p for p in runs_dir.iterdir() if p.is_dir() and token in p.name]
    if not matches:
        raise FileNotFoundError(f"No run directory matches token '{token}' in {runs_dir}")
    if len(matches) > 1:
        names = ", ".join(sorted(p.name for p in matches))
        raise ValueError(f"Token '{token}' is ambiguous; matched: {names}")

    sub = matches[0] / "submission.csv"
    if not sub.is_file():
        raise FileNotFoundError(f"Matched run has no submission.csv: {matches[0]}")
    return sub


def load_submission(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {"id", "target"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    out = df[["id", "target"]].copy()
    out["id"] = out["id"].astype(int)
    out["target"] = out["target"].astype(float)
    return out.sort_values("id").reset_index(drop=True)


In [ ]:
sub_a_path = resolve_submission_source(RUN_A_TOKEN, RUNS_DIR)
sub_b_path = resolve_submission_source(RUN_B_TOKEN, RUNS_DIR)

print("Resolved submission A:", sub_a_path)
print("Resolved submission B:", sub_b_path)

sub_a = load_submission(sub_a_path).rename(columns={"target": "pred_a"})
sub_b = load_submission(sub_b_path).rename(columns={"target": "pred_b"})

ctx = pd.read_csv(TEST_PATH, usecols=["id", "market"])
ctx["id"] = ctx["id"].astype(int)

cmp = (
    ctx
    .merge(sub_a, on="id", how="left", validate="one_to_one")
    .merge(sub_b, on="id", how="left", validate="one_to_one")
    .sort_values(["market", "id"])
    .reset_index(drop=True)
)

if cmp[["pred_a", "pred_b"]].isna().any().any():
    missing_rows = int(cmp[["pred_a", "pred_b"]].isna().any(axis=1).sum())
    raise ValueError(f"Found {missing_rows} rows with missing predictions after merge.")

cmp["diff_b_minus_a"] = cmp["pred_b"] - cmp["pred_a"]
cmp["abs_diff"] = cmp["diff_b_minus_a"].abs()

cmp.head()


In [ ]:
overall = {
    "rows": int(len(cmp)),
    "mean_pred_a": float(cmp["pred_a"].mean()),
    "mean_pred_b": float(cmp["pred_b"].mean()),
    "mean_diff_b_minus_a": float(cmp["diff_b_minus_a"].mean()),
    "mean_abs_diff": float(cmp["abs_diff"].mean()),
    "rmse_diff": float(np.sqrt(np.mean(np.square(cmp["diff_b_minus_a"])))),
    "corr_a_b": float(cmp["pred_a"].corr(cmp["pred_b"])),
}

display(pd.DataFrame([overall]))


In [ ]:
def _market_metrics(g: pd.DataFrame) -> pd.Series:
    diff = g["diff_b_minus_a"].to_numpy(dtype=float)
    return pd.Series(
        {
            "n": int(len(g)),
            "mean_pred_a": float(g["pred_a"].mean()),
            "mean_pred_b": float(g["pred_b"].mean()),
            "mean_diff_b_minus_a": float(g["diff_b_minus_a"].mean()),
            "mean_abs_diff": float(g["abs_diff"].mean()),
            "rmse_diff": float(np.sqrt(np.mean(np.square(diff)))),
            "corr_a_b": float(g["pred_a"].corr(g["pred_b"])),
            "pct_b_higher": float((g["diff_b_minus_a"] > 0).mean()),
        }
    )


per_market = (
    cmp.groupby("market", dropna=False)
    .apply(_market_metrics)
    .reset_index()
    .sort_values("mean_abs_diff", ascending=False)
)

display(per_market)


In [ ]:
fig = px.bar(
    per_market,
    x="market",
    y="mean_abs_diff",
    text="mean_abs_diff",
    title=f"Mean |prediction diff| by market ({RUN_B_TOKEN} - {RUN_A_TOKEN})",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(yaxis_title="mean absolute difference")
fig.show()

fig = px.bar(
    per_market,
    x="market",
    y="mean_diff_b_minus_a",
    text="mean_diff_b_minus_a",
    title=f"Signed mean diff by market ({RUN_B_TOKEN} - {RUN_A_TOKEN})",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(yaxis_title="mean signed difference")
fig.show()


In [ ]:
markets = sorted(cmp["market"].dropna().unique())
if not markets:
    raise ValueError("No market labels found after merge")

fig = make_subplots(
    rows=len(markets),
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.03,
    subplot_titles=[str(m) for m in markets],
)

for i, market in enumerate(markets, start=1):
    d = cmp[cmp["market"] == market].sort_values("id")
    fig.add_trace(
        go.Scatter(
            x=d["id"],
            y=d["pred_a"],
            mode="lines",
            name=f"A ({RUN_A_TOKEN})",
            legendgroup="a",
            showlegend=(i == 1),
            line=dict(width=1.1, color="#1f77b4"),
        ),
        row=i,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=d["id"],
            y=d["pred_b"],
            mode="lines",
            name=f"B ({RUN_B_TOKEN})",
            legendgroup="b",
            showlegend=(i == 1),
            line=dict(width=1.1, color="#ff7f0e"),
        ),
        row=i,
        col=1,
    )

fig.update_layout(
    title=f"Submission prediction overlay per market: {RUN_A_TOKEN} vs {RUN_B_TOKEN}",
    height=max(500, 260 * len(markets)),
)
fig.update_xaxes(title_text="id")
fig.update_yaxes(title_text="target")
fig.show()


In [ ]:
top_disagreement = (
    cmp.sort_values(["market", "abs_diff"], ascending=[True, False])
    .groupby("market", as_index=False)
    .head(20)
    [["market", "id", "pred_a", "pred_b", "diff_b_minus_a", "abs_diff"]]
    .sort_values(["market", "abs_diff"], ascending=[True, False])
)

display(top_disagreement)

fig = px.box(
    cmp,
    x="market",
    y="diff_b_minus_a",
    points=False,
    title=f"Distribution of prediction deltas by market ({RUN_B_TOKEN} - {RUN_A_TOKEN})",
)
fig.update_layout(yaxis_title="prediction delta")
fig.show()
